# EEGModels 演示

这个 notebook 演示了 ARL EEGModels 项目中的深度学习模型，用于 EEG 信号处理和分类。

## 项目概述

ARL EEGModels 是由美国陆军研究实验室开发的，用于 EEG 信号处理的卷积神经网络（CNN）模型集合。

### 主要特点：
- 提供经过验证的 CNN 模型用于 EEG 信号处理和分类
- 促进可重复研究
- 使其他研究者能够轻松使用和比较这些模型

### 实现的模型：
1. **EEGNet** - 紧凑的 CNN 架构，专为 EEG-BCI 设计
2. **EEGNet_SSVEP** - 用于稳态视觉诱发电位（SSVEP）分类的 EEGNet 变体
3. **DeepConvNet** - 深度卷积网络
4. **ShallowConvNet** - 浅层卷积网络

### 参考文献：
- EEGNet: Lawhern et al. (2018). *Journal of Neural Engineering*, 15(5), 056013
- DeepConvNet/ShallowConvNet: Schirrmeister et al. (2017). *Human Brain Mapping*, 38(11), 5391-5420

## 1. 环境设置

首先导入必要的库和模型。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import sys

sys.path.append('/Users/hyijie/Desktop/ai/ai_手搓EEGnet/arl-eegmodels')
from EEGModels import EEGNet, EEGNet_SSVEP, DeepConvNet, ShallowConvNet

print("导入成功！")
print(f"TensorFlow 版本: {__import__('tensorflow').__version__}")

ModuleNotFoundError: No module named 'numpy'

## 2. 生成模拟 EEG 数据

为了演示，我们生成一些模拟的 EEG 数据。

In [ ]:
def generate_eeg_data(n_samples=1000, n_channels=64, n_timepoints=128, n_classes=4):
    """
    生成模拟的 EEG 数据
    
    参数:
        n_samples: 样本数量
        n_channels: EEG 通道数
        n_timepoints: 时间点数
        n_classes: 分类类别数
    
    返回:
        X: EEG 数据 (n_samples, n_channels, n_timepoints, 1)
        y: 标签 (n_samples,)
    """
    np.random.seed(42)
    
    # 生成随机 EEG 数据
    X = np.random.randn(n_samples, n_channels, n_timepoints)
    
    # 添加一些类别特定的特征
    for i in range(n_samples):
        class_id = i % n_classes
        # 为每个类别添加特定的频率成分
        freq = 10 + class_id * 5  # 不同类别不同频率
        t = np.linspace(0, 1, n_timepoints)
        signal = np.sin(2 * np.pi * freq * t)
        X[i, :n_channels//2, :] += 0.5 * signal
    
    # 添加通道维度
    X = X.reshape(X.shape[0], n_channels, n_timepoints, 1)
    
    # 生成标签
    y = np.random.randint(0, n_classes, n_samples)
    
    return X, y

# 生成数据
n_samples = 1000
n_channels = 64
n_timepoints = 128
n_classes = 4

X, y = generate_eeg_data(n_samples, n_channels, n_timepoints, n_classes)

print(f"数据形状: X = {X.shape}, y = {y.shape}")
print(f"类别分布: {np.bincount(y)}")

## 3. 数据预处理

将数据分为训练集和测试集，并进行 one-hot 编码。

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 将标签转换为 one-hot 编码
y_train_onehot = to_categorical(y_train, n_classes)
y_test_onehot = to_categorical(y_test, n_classes)

print(f"训练集: X_train = {X_train.shape}, y_train = {y_train_onehot.shape}")
print(f"测试集: X_test = {X_test.shape}, y_test = {y_test_onehot.shape}")

## 4. EEGNet 模型

EEGNet 是一个紧凑的 CNN 架构，专为 EEG-BCI 设计。它使用深度可分离卷积来学习空间和时间特征。

In [ ]:
# 创建 EEGNet 模型
model_eegnet = EEGNet(
    nb_classes=n_classes,
    Chans=n_channels,
    Samples=n_timepoints,
    dropoutRate=0.5,
    kernLength=64,
    F1=8,
    D=2,
    F2=16,
    norm_rate=0.25,
    dropoutType='Dropout'
)

# 编译模型
model_eegnet.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# 显示模型架构
model_eegnet.summary()

In [ ]:
# 训练 EEGNet 模型
print("训练 EEGNet 模型...")
history_eegnet = model_eegnet.fit(
    X_train, y_train_onehot,
    batch_size=32,
    epochs=10,
    validation_data=(X_test, y_test_onehot),
    verbose=1
)

# 评估模型
test_loss, test_acc = model_eegnet.evaluate(X_test, y_test_onehot, verbose=0)
print(f"\nEEGNet 测试准确率: {test_acc:.4f}")

## 5. DeepConvNet 模型

DeepConvNet 是一个深度卷积网络，包含多个卷积层和池化层。

In [ ]:
# 创建 DeepConvNet 模型
model_deep = DeepConvNet(
    nb_classes=n_classes,
    Chans=n_channels,
    Samples=n_timepoints,
    dropoutRate=0.5
)

# 编译模型
model_deep.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# 显示模型架构
model_deep.summary()

In [ ]:
# 训练 DeepConvNet 模型
print("训练 DeepConvNet 模型...")
history_deep = model_deep.fit(
    X_train, y_train_onehot,
    batch_size=32,
    epochs=10,
    validation_data=(X_test, y_test_onehot),
    verbose=1
)

# 评估模型
test_loss, test_acc = model_deep.evaluate(X_test, y_test_onehot, verbose=0)
print(f"\nDeepConvNet 测试准确率: {test_acc:.4f}")

## 6. ShallowConvNet 模型

ShallowConvNet 是一个浅层卷积网络，使用平方和对数激活函数。

In [ ]:
# 创建 ShallowConvNet 模型
model_shallow = ShallowConvNet(
    nb_classes=n_classes,
    Chans=n_channels,
    Samples=n_timepoints,
    dropoutRate=0.5
)

# 编译模型
model_shallow.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# 显示模型架构
model_shallow.summary()

In [ ]:
# 训练 ShallowConvNet 模型
print("训练 ShallowConvNet 模型...")
history_shallow = model_shallow.fit(
    X_train, y_train_onehot,
    batch_size=32,
    epochs=10,
    validation_data=(X_test, y_test_onehot),
    verbose=1
)

# 评估模型
test_loss, test_acc = model_shallow.evaluate(X_test, y_test_onehot, verbose=0)
print(f"\nShallowConvNet 测试准确率: {test_acc:.4f}")

## 7. 模型比较

比较不同模型的性能。

In [ ]:
# 绘制训练历史
plt.figure(figsize=(15, 5))

# 准确率曲线
plt.subplot(1, 2, 1)
plt.plot(history_eegnet.history['accuracy'], label='EEGNet Train', linewidth=2)
plt.plot(history_eegnet.history['val_accuracy'], label='EEGNet Val', linewidth=2)
plt.plot(history_deep.history['accuracy'], label='DeepConvNet Train', linewidth=2)
plt.plot(history_deep.history['val_accuracy'], label='DeepConvNet Val', linewidth=2)
plt.plot(history_shallow.history['accuracy'], label='ShallowConvNet Train', linewidth=2)
plt.plot(history_shallow.history['val_accuracy'], label='ShallowConvNet Val', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('模型准确率比较')
plt.legend()
plt.grid(True, alpha=0.3)

# 损失曲线
plt.subplot(1, 2, 2)
plt.plot(history_eegnet.history['loss'], label='EEGNet Train', linewidth=2)
plt.plot(history_eegnet.history['val_loss'], label='EEGNet Val', linewidth=2)
plt.plot(history_deep.history['loss'], label='DeepConvNet Train', linewidth=2)
plt.plot(history_deep.history['val_loss'], label='DeepConvNet Val', linewidth=2)
plt.plot(history_shallow.history['loss'], label='ShallowConvNet Train', linewidth=2)
plt.plot(history_shallow.history['val_loss'], label='ShallowConvNet Val', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('模型损失比较')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 比较最终性能
models = {
    'EEGNet': model_eegnet,
    'DeepConvNet': model_deep,
    'ShallowConvNet': model_shallow
}

print("模型性能比较:")
print("-" * 50)
for name, model in models.items():
    train_loss, train_acc = model.evaluate(X_train, y_train_onehot, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test_onehot, verbose=0)
    print(f"{name:15s} - 训练准确率: {train_acc:.4f}, 测试准确率: {test_acc:.4f}")
    
    # 计算参数数量
    total_params = model.count_params()
    print(f"{name:15s} - 参数数量: {total_params:,}")

## 8. 预测示例

使用训练好的模型进行预测。

In [ ]:
# 使用 EEGNet 进行预测
predictions = model_eegnet.predict(X_test[:5])
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test_onehot[:5], axis=1)

print("预测结果示例:")
print("-" * 50)
for i in range(5):
    print(f"样本 {i+1}: 真实类别 = {true_classes[i]}, 预测类别 = {predicted_classes[i]}")
    print(f"  概率分布: {predictions[i]}")
    print()

## 9. 可视化 EEG 数据

可视化一些 EEG 样本。

In [ ]:
# 可视化 EEG 数据
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i in range(4):
    ax = axes[i]
    # 绘制前 10 个通道
    for ch in range(10):
        ax.plot(X_test[i, ch, :, 0], alpha=0.7, label=f'Ch {ch+1}')
    
    ax.set_title(f'样本 {i+1} - 类别 {true_classes[i]}')
    ax.set_xlabel('时间点')
    ax.set_ylabel('振幅')
    ax.legend(loc='upper right', fontsize='small')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. EEGNet_SSVEP 模型

EEGNet_SSVEP 是专门用于稳态视觉诱发电位（SSVEP）分类的 EEGNet 变体。

In [ ]:
# 创建 SSVEP 数据（通常 SSVEP 使用较少的通道和更多的时间点）
n_channels_ssvep = 8
n_timepoints_ssvep = 256
n_classes_ssvep = 12

X_ssvep, y_ssvep = generate_eeg_data(
    n_samples=500,
    n_channels=n_channels_ssvep,
    n_timepoints=n_timepoints_ssvep,
    n_classes=n_classes_ssvep
)

# 划分数据
X_train_ssvep, X_test_ssvep, y_train_ssvep, y_test_ssvep = train_test_split(
    X_ssvep, y_ssvep, test_size=0.2, random_state=42, stratify=y_ssvep
)

y_train_ssvep_onehot = to_categorical(y_train_ssvep, n_classes_ssvep)
y_test_ssvep_onehot = to_categorical(y_test_ssvep, n_classes_ssvep)

print(f"SSVEP 数据形状: X_train = {X_train_ssvep.shape}, y_train = {y_train_ssvep_onehot.shape}")

In [ ]:
# 创建 EEGNet_SSVEP 模型
model_ssvep = EEGNet_SSVEP(
    nb_classes=n_classes_ssvep,
    Chans=n_channels_ssvep,
    Samples=n_timepoints_ssvep,
    dropoutRate=0.5,
    kernLength=256,
    F1=96,
    D=1,
    F2=96,
    dropoutType='Dropout'
)

# 编译模型
model_ssvep.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# 显示模型架构
model_ssvep.summary()

In [ ]:
# 训练 EEGNet_SSVEP 模型
print("训练 EEGNet_SSVEP 模型...")
history_ssvep = model_ssvep.fit(
    X_train_ssvep, y_train_ssvep_onehot,
    batch_size=32,
    epochs=10,
    validation_data=(X_test_ssvep, y_test_ssvep_onehot),
    verbose=1
)

# 评估模型
test_loss, test_acc = model_ssvep.evaluate(X_test_ssvep, y_test_ssvep_onehot, verbose=0)
print(f"\nEEGNet_SSVEP 测试准确率: {test_acc:.4f}")

## 11. 模型参数调优建议

### EEGNet 参数：
- **F1**: 时间滤波器数量（默认 8）
- **D**: 每个时间卷积中的空间滤波器数量（默认 2）
- **F2**: 点积滤波器数量（默认 F1 * D）
- **kernLength**: 第一层时间卷积的长度（建议设置为采样率的一半）
- **dropoutType**: 'Dropout' 或 'SpatialDropout2D'

### DeepConvNet/ShallowConvNet 参数：
- 主要参数是通道数（Chans）和时间点数（Samples）
- 这些模型假设输入是 128Hz 采样的 2 秒 EEG 信号

### 调优建议：
1. 对于小数据集，使用较小的 F1 和 D 值
2. 对于高采样率数据，增加 kernLength
3. 使用 SpatialDropout2D 可能对 ERP 信号分类效果更好
4. 进行网格搜索以找到最佳参数组合

## 12. 保存和加载模型

演示如何保存和加载训练好的模型。

In [ ]:
# 保存模型
model_eegnet.save('eegnet_model.h5')
print("模型已保存为 eegnet_model.h5")

# 加载模型
from tensorflow.keras.models import load_model
loaded_model = load_model('eegnet_model.h5')
print("模型已加载")

# 验证加载的模型
test_loss, test_acc = loaded_model.evaluate(X_test, y_test_onehot, verbose=0)
print(f"加载模型的测试准确率: {test_acc:.4f}")

## 总结

本 notebook 演示了 ARL EEGModels 项目中的四个主要模型：

1. **EEGNet**: 紧凑高效的 CNN 架构，适合大多数 EEG 分类任务
2. **EEGNet_SSVEP**: 专门用于 SSVEP 信号分类
3. **DeepConvNet**: 深度卷积网络，适合复杂模式识别
4. **ShallowConvNet**: 浅层卷积网络，计算效率高

### 关键要点：
- 所有模型都使用 Keras 和 TensorFlow 实现
- 模型输入格式为 (Chans, Samples, 1)
- 支持多类别分类任务
- 可以通过调整参数来适应不同的数据集

### 下一步：
- 在真实 EEG 数据上测试这些模型
- 进行超参数调优
- 尝试不同的数据增强技术
- 探索模型解释方法（如 DeepExplain）